In [3]:
import sys

# Point Python at the folder with the .py files
sys.path.insert(0, r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\ctr_loader")

# Force reload in case you re-edit any of the files
for m in ["products_registry", "basis_loader", "assumption_loader"]:
    if m in sys.modules:
        del sys.modules[m]

from basis_loader import load_basis
from assumption_loader import load_assumptions
from products_registry import PRODUCT_REGISTRY, resolve_ci_mortality_table

BASIS_SNAP  = r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07"
ASSUMP_SNAP = r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\assumption_snapshots\assumption_A2026_07"

basis  = load_basis(BASIS_SNAP)
assump = load_assumptions(ASSUMP_SNAP)

print("=" * 70)
print(f"basis tables ({len(basis.tables)}):")
for name, df in basis.tables.items():
    print(f"  {name:<22} {df.shape}")

print(f"\nassumption tables ({len(assump.tables)}):")
for name, df in assump.tables.items():
    print(f"  {name:<22} {df.shape}")

print(f"\nassump.globals ({len(assump.globals)} keys):")
for k, v in assump.globals.items():
    print(f"  {k:<32} = {v}")

print("\n" + "=" * 70)
print("BASIS for_product() TESTS")
print("=" * 70)
for pkey in ["gla", "lsdb", "ci", "spouse_gla", "terminal", "accident_death"]:
    slices = basis.for_product(pkey)
    print(f"\n  {pkey}:")
    for sname, df in slices.items():
        prods = ""
        if "product" in df.columns:
            prods = f"  products={sorted(df['product'].unique().to_list())}"
        print(f"    {sname:<20} shape={df.shape}{prods}")

print("\n" + "=" * 70)
print("ASSUMPTION for_product() TESTS")
print("=" * 70)
for pkey in ["gla", "lsdb", "ci", "spouse_gla", "terminal"]:
    slices = assump.for_product(pkey)
    print(f"\n  {pkey}:")
    for sname, df in slices.items():
        prods = ""
        if "product" in df.columns:
            prods = f"  products={sorted(df['product'].unique().to_list())}"
        print(f"    {sname:<20} shape={df.shape}{prods}")

print("\n" + "=" * 70)
print("CI MORTALITY RESOLVER")
print("=" * 70)
ci_map = assump.tables["ci_table_mapping"]
mort_products = set(basis.tables["mortality"]["product"].unique().to_list())
for opt, var in [
    ("Comprehensive_Plus", "accelerator"),
    ("Core",               "accelerator"),
    ("Core",               "standalone"),
    ("Cancer_Protect",     "accelerator"),
    ("Comprehensive",      "standalone"),
    ("Extensive",          "accelerator"),
]:
    r = resolve_ci_mortality_table(ci_map, opt, var)
    ok = "OK" if r["table_key"] in mort_products else "MISSING"
    print(f"  {opt:<22} + {var:<12} -> {r['table_key']:<32} "
          f"age_def={r['age_definition']} variant={r['variant_type']}  [{ok}]")

print("\nDONE.")

basis tables (8):
  industry               (368, 10)
  mortality              (2160, 5)
  product_type_adj       (2, 3)
  regional_factors       (23, 3)
  salary_band            (102, 5)
  ti_factors             (195, 6)
  trend                  (66, 5)
  waiting_periods        (5, 3)

assumption tables (10):
  ci_reinstatement       (17, 6)
  ci_table_mapping       (7, 7)
  conversion_option      (11, 8)
  ctc_options            (3, 6)
  globals                (10, 6)
  lsdb_features          (87, 7)
  product_limits         (53, 8)
  profit_margins         (9, 7)
  spouse_factors         (6, 7)
  tax_tables             (4, 9)

assump.globals (10 keys):
  vat_rate                         = 0.15
  active_country_code              = 1.0
  namibia_competitive_factor       = -0.35
  namibia_stamps_duty              = 0.03
  namibia_covid_loading_factor     = 
  eac_profit_margin                = 0.06
  eac_admin_adjustment             = 1.005
  eac_optional_cover_margin        = 0.15
  ea

In [10]:
# ============================================================
#  FULL 3-WAY INTEGRATION TEST — CTR + BASIS + ASSUMPTIONS
# ============================================================
import sys, importlib, polars as pl
from pathlib import Path

sys.path.insert(0, r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\ctr_loader")

# Reload everything fresh
for m in ["products_registry", "ctr_coerce", "ctr_schemas",
          "ctr_loader_v2", "basis_loader", "assumption_loader"]:
    if m in sys.modules:
        del sys.modules[m]

from products_registry import PRODUCT_REGISTRY, resolve_ci_mortality_table
from basis_loader import load_basis
from assumption_loader import load_assumptions
from ctr_loader_v2 import load_ctr

BASIS_SNAP  = r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07"
ASSUMP_SNAP = r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\assumption_snapshots\assumption_A2026_07"
CTR_PATH    = r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\Tests MH\CTRs\De Beers Group Services (Pty) Ltd_Quote1 - Copy.xls" # <-- adjust if needed

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(50)

# ------------------------------------------------------------
# 1. Load all three
# ------------------------------------------------------------
print("=" * 80)
print("STEP 1 — LOAD ALL THREE")
print("=" * 80)
basis  = load_basis(BASIS_SNAP)
assump = load_assumptions(ASSUMP_SNAP)
ctr    = load_ctr(CTR_PATH)
print(f"  ✓ basis  : {len(basis.tables)} tables")
print(f"  ✓ assump : {len(assump.tables)} tables, {len(assump.globals)} globals")
print(f"  ✓ ctr    : {len(ctr.active_products)} active products, "
      f"{ctr.categories.height} category-field rows, "
      f"{sum(df.height for df in ctr.members.values())} members")

# ------------------------------------------------------------
# 2. Verify each ACTIVE product resolves cleanly across all 3
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 2 — PER-PRODUCT 3-WAY SLICE CHECK")
print("=" * 80)
summary_rows = []
for pkey in ctr.active_products:
    spec = PRODUCT_REGISTRY[pkey]
    # CTR side
    ctr_cats = ctr.categories.filter(pl.col("product") == pkey)
    n_cats   = ctr_cats["category_number"].n_unique() if ctr_cats.height else 0
    variant  = ctr_cats["variant"].unique().to_list() if ctr_cats.height else []
    member_src = spec.member_source
    role = "gla_sheet" if member_src.endswith("- GLA") else "sp_sheet"
    n_members = ctr.members.get(role, pl.DataFrame()).height

    # Basis + Assump side
    b_slice = basis.for_product(pkey)
    a_slice = assump.for_product(pkey)

    summary_rows.append({
        "product": pkey,
        "ctr_cats": n_cats,
        "ctr_variant": ",".join(variant),
        "ctr_members": n_members,
        "basis_tables": len(b_slice),
        "assump_tables": len(a_slice),
        "basis_missing": _flag_missing_basis(spec, b_slice),
        "assump_missing": _flag_missing_assump(spec, a_slice),
    })

def _flag_missing_basis(spec, sl):
    exp = []
    if spec.basis_mortality:        exp.append("mortality")
    if spec.basis_salary_band:      exp.append("salary_band")
    if spec.basis_industry:         exp.append("industry")
    if spec.basis_regional:         exp.append("regional_factors")
    if spec.basis_trend:            exp.append("trend")
    if spec.basis_waiting_periods:  exp.append("waiting_periods")
    if spec.basis_product_type_adj: exp.append("product_type_adj")
    if spec.uses_ti_factors:        exp.append("ti_factors")
    return [e for e in exp if e not in sl]

def _flag_missing_assump(spec, sl):
    exp = []
    if spec.assumption_limits:      exp.append("product_limits")
    if spec.assumption_margins:     exp.append("profit_margins")
    if spec.assumption_co:          exp.append("conversion_option")
    if spec.uses_lsdb_features:     exp.append("lsdb_features")
    if spec.uses_spouse_factors:    exp.append("spouse_factors")
    if spec.uses_ci_table_mapping:  exp.append("ci_table_mapping")
    if spec.uses_ci_reinstatement:  exp.append("ci_reinstatement")
    if spec.uses_ctc_options:       exp.append("ctc_options")
    if spec.uses_tax_tables:        exp.append("tax_tables")
    return [e for e in exp if e not in sl]

# Recompute using the helpers (function order fix)
summary_rows = []
for pkey in ctr.active_products:
    spec = PRODUCT_REGISTRY[pkey]
    ctr_cats = ctr.categories.filter(pl.col("product") == pkey)
    n_cats   = ctr_cats["category_number"].n_unique() if ctr_cats.height else 0
    variant  = ctr_cats["variant"].unique().to_list() if ctr_cats.height else []
    role = "gla_sheet" if spec.member_source.endswith("- GLA") else "sp_sheet"
    n_members = ctr.members.get(role, pl.DataFrame()).height
    b_slice = basis.for_product(pkey)
    a_slice = assump.for_product(pkey)
    b_miss = _flag_missing_basis(spec, b_slice)
    a_miss = _flag_missing_assump(spec, a_slice)
    summary_rows.append({
        "product": pkey,
        "cats": n_cats,
        "variant": ",".join(variant) or "-",
        "members": n_members,
        "basis_tbls": len(b_slice),
        "assump_tbls": len(a_slice),
        "basis_missing":  ",".join(b_miss)  or "✓",
        "assump_missing": ",".join(a_miss) or "✓",
    })
print(pl.DataFrame(summary_rows))

# ------------------------------------------------------------
# 3. WORKED EXAMPLE — GLA, member #1 (Jayden Davis), Category 1
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 3 — WORKED EXAMPLE: GLA, member #1, Category 1")
print("=" * 80)
gla = ctr.members["gla_sheet"].head(1)
print("Member row:")
print(gla.select(["member_name","gender","date_of_birth","annual_salary","category_flag","status"]))

# --- CTR side: GLA Cat 1 config ---
gla_c1 = ctr.categories.filter(
    (pl.col("product") == "gla") & (pl.col("category_number") == 1)
)
print("\nCTR GLA Category 1 config:")
print(gla_c1.select(["field","value"]))

# --- Basis side: GLA slice ---
b = basis.for_product("gla")
member = gla.row(0, named=True)
gender = member["gender"]
salary = member["annual_salary"]

# Age at calculation date
from datetime import date
calc_date = ctr.scheme.calculation_date
dob = member["date_of_birth"]
age = calc_date.year - dob.year - ((calc_date.month, calc_date.day) < (dob.month, dob.day))
print(f"\nComputed age at {calc_date}: {age}")

# Mortality rate (base + AIDS)
mort_base = b["mortality"].filter(
    (pl.col("product") == "gla") & (pl.col("age") == age) & (pl.col("gender") == gender)
)
mort_aids = b["mortality"].filter(
    (pl.col("product") == "gla_aids") & (pl.col("age") == age) & (pl.col("gender") == gender)
)
print(f"\nBasis: GLA mortality  age={age} {gender}:")
print(mort_base)
print(f"Basis: GLA AIDS       age={age} {gender}:")
print(mort_aids)

# Salary band factor
sb = b["salary_band"].filter(
    (pl.col("product") == "gla") & (pl.col("gender") == gender)
).sort("salary_from")
# find the band this salary falls into
band = sb.filter(pl.col("salary_from") <= salary).tail(1)
print(f"\nBasis: GLA salary band for salary {salary:,.2f} ({gender}):")
print(band)

# Trend factor for calc year
trend = b["trend"].filter(
    (pl.col("product") == "gla") & (pl.col("gender") == gender)
).sort("year")
print(f"\nBasis: GLA trend rows for {gender} (last 3 years):")
print(trend.tail(3))

# Industry factor for scheme's primary
industry = b["industry"].filter(
    (pl.col("product") == "gla") &
    (pl.col("primary") == ctr.scheme.regions.get("primary") if False else True)
).filter(pl.col("primary") == "Agriculture, Forestry and Fishing")
print(f"\nBasis: GLA industry rows for 'Agriculture, Forestry and Fishing':")
print(industry.select(["primary","secondary","grouping","scheme_size","inc_adj","acaf_adj"]))

# --- Assumption side: GLA slice ---
a = assump.for_product("gla")
print(f"\nAssump: GLA product_limits:")
print(a["product_limits"])
print(f"\nAssump: GLA profit_margins:")
print(a["profit_margins"])
print(f"\nAssump: GLA conversion_option:")
print(a["conversion_option"])
print(f"\nAssump: SA tax_tables:")
print(a["tax_tables"])
print(f"\nAssump globals (vat_rate, scheme_size_large_threshold):")
print(f"  vat_rate = {assump.globals['vat_rate']}")
print(f"  scheme_size_large_threshold = {assump.globals['scheme_size_large_threshold']}")

# ------------------------------------------------------------
# 4. WORKED EXAMPLE — CI, Category 1 dynamic table resolution
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 4 — WORKED EXAMPLE: CI Category 1 (dynamic mortality resolution)")
print("=" * 80)

# Get product_option from CTR
ci_c1 = ctr.categories.filter(
    (pl.col("product") == "ci") & (pl.col("category_number") == 1)
)
prod_opt = ci_c1.filter(pl.col("field") == "product_option")["value"][0]
variant  = ci_c1["variant"][0]
rating   = ctr.scheme.rating.get("ci")
print(f"CTR: CI Cat 1 product_option = {prod_opt}")
print(f"CTR: CI Cat 1 variant        = {variant}")
print(f"CTR: scheme rating[ci]       = {rating}")

# Resolve mortality table
resolved = resolve_ci_mortality_table(
    assump.tables["ci_table_mapping"], prod_opt, variant
)
print(f"\nResolved mortality: {resolved}")

# Pull the rate for age 40 M/F
key = resolved["table_key"]
ci_mort = basis.tables["mortality"].filter(
    (pl.col("product") == key) & (pl.col("age").is_in([40, 50, 60]))
).sort(["age","gender"])
print(f"\nBasis: mortality[{key}] at ages 40/50/60:")
print(ci_mort)

# ------------------------------------------------------------
# 5. WORKED EXAMPLE — LSDB, features + waiting periods
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 5 — WORKED EXAMPLE: LSDB, features + waiting periods")
print("=" * 80)

lsdb_c1 = ctr.categories.filter(
    (pl.col("product") == "lsdb") & (pl.col("category_number") == 1)
)
wp   = lsdb_c1.filter(pl.col("field") == "waiting_period")["value"][0]
defn = lsdb_c1.filter(pl.col("field") == "definition_of_disability")["value"][0]
print(f"CTR: LSDB Cat 1 waiting_period = {wp}")
print(f"CTR: LSDB Cat 1 definition     = {defn}")

lb = basis.for_product("lsdb")
la = assump.for_product("lsdb")

# waiting_periods factor
wp_match = lb["waiting_periods"].filter(pl.col("wp_months") == float(wp))
print(f"\nBasis: LSDB waiting_periods for wp_months={wp}:")
print(wp_match)

# db_definitions factor from lsdb_features
db_defs = la["lsdb_features"].filter(pl.col("feature") == "db_definitions")
print(f"\nAssump: lsdb_features db_definitions:")
print(db_defs)

# ------------------------------------------------------------
# 6. WORKED EXAMPLE — Spouse (shared basis, split limits)
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 6 — WORKED EXAMPLE: Spouse GLA vs Spouse LSDB")
print("=" * 80)

for pk in ["spouse_gla", "spouse_lsdb"]:
    b = basis.for_product(pk)
    a = assump.for_product(pk)
    print(f"\n{pk}:")
    print(f"  basis  mortality shape = {b['mortality'].shape}  "
          f"products={sorted(b['mortality']['product'].unique().to_list())}")
    print(f"  assump product_limits:")
    print(a["product_limits"])

# ------------------------------------------------------------
# 7. WORKED EXAMPLE — Terminal Illness (uses ti_factors)
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 7 — WORKED EXAMPLE: Terminal Illness (ti_factors)")
print("=" * 80)
b = basis.for_product("terminal")
a = assump.for_product("terminal")
print("Basis ti_factors head at age 40:")
print(b["ti_factors"].filter(pl.col("age") == 40))
print("\nAssump product_limits:")
print(a["product_limits"])
print("\nAssump profit_margins:")
print(a["profit_margins"])

# ------------------------------------------------------------
# 8. Sanity: any active product with ZERO basis or ZERO assump?
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 8 — COMPLETENESS SANITY")
print("=" * 80)
for pkey in ctr.active_products:
    b = basis.for_product(pkey)
    a = assump.for_product(pkey)
    flag_b = "⚠ NO BASIS"  if not b else f"{len(b)} tables"
    flag_a = "⚠ NO ASSUMP" if not a else f"{len(a)} tables"
    print(f"  {pkey:<22} basis={flag_b:<18} assump={flag_a}")

print("\n" + "=" * 80)
print("ALL TESTS COMPLETE.")
print("=" * 80)

STEP 1 — LOAD ALL THREE
  ✓ Scheme Details - Save parsed in 0.00s
  ✓ CatInput-Save parsed in 0.01s  (486 rows)
  ✓ Member Data - GLA parsed in 0.03s  (424 members)
  ✓ Member Data - SP parsed in 0.02s  (424 members)
  ⏱  Total CTR load: 7.99s
  ✓ basis  : 8 tables
  ✓ assump : 10 tables, 10 globals
  ✓ ctr    : 9 active products, 486 category-field rows, 848 members

STEP 2 — PER-PRODUCT 3-WAY SLICE CHECK
shape: (9, 8)
┌─────────────────────┬──────┬─────────────┬─────────┬────────────┬─────────────┬───────────────┬────────────────┐
│ product             ┆ cats ┆ variant     ┆ members ┆ basis_tbls ┆ assump_tbls ┆ basis_missing ┆ assump_missing │
│ ---                 ┆ ---  ┆ ---         ┆ ---     ┆ ---        ┆ ---         ┆ ---           ┆ ---            │
│ str                 ┆ i64  ┆ str         ┆ i64     ┆ i64        ┆ i64         ┆ str           ┆ str            │
╞═════════════════════╪══════╪═════════════╪═════════╪════════════╪═════════════╪═══════════════╪════════════════╡
│

In [9]:
# ------------------------------------------------------------
# 2. Verify each ACTIVE product resolves cleanly across all 3
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 2 — PER-PRODUCT 3-WAY SLICE CHECK")
print("=" * 80)

def _flag_missing_basis(spec, sl):
    exp = []
    if spec.basis_mortality:        exp.append("mortality")
    if spec.basis_salary_band:      exp.append("salary_band")
    if spec.basis_industry:         exp.append("industry")
    if spec.basis_regional:         exp.append("regional_factors")
    if spec.basis_trend:            exp.append("trend")
    if spec.basis_waiting_periods:  exp.append("waiting_periods")
    if spec.basis_product_type_adj: exp.append("product_type_adj")
    if spec.uses_ti_factors:        exp.append("ti_factors")
    return [e for e in exp if e not in sl]

def _flag_missing_assump(spec, sl):
    exp = []
    if spec.assumption_limits:      exp.append("product_limits")
    if spec.assumption_margins:     exp.append("profit_margins")
    if spec.assumption_co:          exp.append("conversion_option")
    if spec.uses_lsdb_features:     exp.append("lsdb_features")
    if spec.uses_spouse_factors:    exp.append("spouse_factors")
    if spec.uses_ci_table_mapping:  exp.append("ci_table_mapping")
    if spec.uses_ci_reinstatement:  exp.append("ci_reinstatement")
    if spec.uses_ctc_options:       exp.append("ctc_options")
    if spec.uses_tax_tables:        exp.append("tax_tables")
    return [e for e in exp if e not in sl]

summary_rows = []
for pkey in ctr.active_products:
    spec = PRODUCT_REGISTRY[pkey]
    ctr_cats = ctr.categories.filter(pl.col("product") == pkey)
    n_cats   = ctr_cats["category_number"].n_unique() if ctr_cats.height else 0
    variant  = ctr_cats["variant"].unique().to_list() if ctr_cats.height else []
    role = "gla_sheet" if spec.member_source.endswith("- GLA") else "sp_sheet"
    n_members = ctr.members.get(role, pl.DataFrame()).height
    b_slice = basis.for_product(pkey)
    a_slice = assump.for_product(pkey)
    b_miss = _flag_missing_basis(spec, b_slice)
    a_miss = _flag_missing_assump(spec, a_slice)
    summary_rows.append({
        "product": pkey,
        "cats": n_cats,
        "variant": ",".join(variant) or "-",
        "members": n_members,
        "basis_tbls": len(b_slice),
        "assump_tbls": len(a_slice),
        "basis_missing":  ",".join(b_miss)  or "OK",
        "assump_missing": ",".join(a_miss) or "OK",
    })
print(pl.DataFrame(summary_rows))


STEP 2 — PER-PRODUCT 3-WAY SLICE CHECK
shape: (9, 8)
┌─────────────────────┬──────┬─────────────┬─────────┬────────────┬─────────────┬───────────────┬────────────────┐
│ product             ┆ cats ┆ variant     ┆ members ┆ basis_tbls ┆ assump_tbls ┆ basis_missing ┆ assump_missing │
│ ---                 ┆ ---  ┆ ---         ┆ ---     ┆ ---        ┆ ---         ┆ ---           ┆ ---            │
│ str                 ┆ i64  ┆ str         ┆ i64     ┆ i64        ┆ i64         ┆ str           ┆ str            │
╞═════════════════════╪══════╪═════════════╪═════════╪════════════╪═════════════╪═══════════════╪════════════════╡
│ gla                 ┆ 6    ┆ default     ┆ 424     ┆ 4          ┆ 4           ┆ OK            ┆ OK             │
│ lsdb                ┆ 6    ┆ accelerator ┆ 424     ┆ 6          ┆ 4           ┆ OK            ┆ OK             │
│ ci                  ┆ 6    ┆ accelerator ┆ 424     ┆ 3          ┆ 5           ┆ OK            ┆ OK             │
│ spouse_gla          ┆ 6 

In [2]:
from pathlib import Path

BASIS_SNAP  = r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07"
ASSUMP_SNAP = r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\assumption_snapshots\assumption_A2026_07"

print("BASIS folder contents:")
for p in sorted(Path(BASIS_SNAP).iterdir()):
    print(f"  {p.name}")

print("\nASSUMPTION folder contents:")
for p in sorted(Path(ASSUMP_SNAP).iterdir()):
    print(f"  {p.name}")

BASIS folder contents:
  build_log.csv
  industry.parquet
  mortality.parquet
  product_type_adj.parquet
  regional_factors.parquet
  salary_band.parquet
  ti_factors.parquet
  trend.parquet
  waiting_periods.parquet

ASSUMPTION folder contents:
  build_log.csv
  ci_reinstatement.parquet
  ci_table_mapping.parquet
  conversion_option.parquet
  ctc_options.parquet
  globals.parquet
  lsdb_features.parquet
  product_limits.parquet
  profit_margins.parquet
  spouse_factors.parquet
  tax_tables.parquet
